# Apache Beam Lab 3: Core Transforms - Solution

## Overview
This notebook contains the solution for Lab 3: Core Transforms.

## Solution Implementation

In [ ]:
import apache_beam as beam

class UppercaseFn(beam.DoFn):
    def process(self, element):
        yield element.upper()

def basic_pardo():
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create words' >> beam.Create(['hello', 'world', 'beam'])
            | 'Uppercase' >> beam.ParDo(UppercaseFn())
            | 'Print' >> beam.Map(print)
        )

print("ParDo concept: Apply custom processing to each element")

## Exercise 1: Custom Data Processing with ParDo

In [ ]:
import pandas as pd
import os

class EnrichSalesFn(beam.DoFn):
    def process(self, element):
        total = element['quantity'] * element['price']
        
        discount = 0.0
        if total > 1000:
            discount = 0.10
        elif total > 500:
            discount = 0.05
        
        enriched = {
            **element,
            'total_price': total,
            'discount_percentage': discount,
            'discounted_total': total * (1 - discount),
            'discount_amount': total * discount
        }
        yield enriched

def enrich_sales_data():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create sales data' >> beam.Create(data)
            | 'Enrich data' >> beam.ParDo(EnrichSalesFn())
            | 'Format output' >> beam.Map(
                lambda x: f"Transaction {x['transaction_id']}: ${x['discounted_total']:.2f} (saved ${x['discount_amount']:.2f})")
            | 'Print' >> beam.Map(print)
        )

print("Enriching sales data with custom ParDo...")
enrich_sales_data()

## GroupByKey Transform

In [ ]:
def group_by_key_example():
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create pairs' >> beam.Create([
                ('apple', 1), ('banana', 2), ('apple', 3), 
                ('orange', 4), ('banana', 5), ('apple', 6)
            ])
            | 'Group by key' >> beam.GroupByKey()
            | 'Format' >> beam.Map(lambda x: f"{x[0]}: {list(x[1])}")
            | 'Print' >> beam.Map(print)
        )

print("GroupByKey concept: Group values by their keys")

## Exercise 2: Sales Aggregation with GroupByKey

In [ ]:
def aggregate_sales_by_key():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Extract product total' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Group by product' >> beam.GroupByKey()
            | 'Sum values' >> beam.Map(lambda x: (x[0], sum(x[1])))
            | 'Format' >> beam.Map(lambda x: f"Product {x[0]}: ${x[1]:.2f}")
            | 'Print' >> beam.Map(print)
        )

print("Aggregating sales data with GroupByKey...")
aggregate_sales_by_key()

## Combine Transforms

In [ ]:
def combine_example():
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create pairs' >> beam.Create([
                ('a', 1), ('b', 2), ('a', 3), ('b', 4), ('a', 5)
            ])
            | 'Combine per key' >> beam.CombinePerKey(sum)
            | 'Print' >> beam.Map(print)
        )
        
        (
            pipeline
            | 'Create numbers' >> beam.Create([1, 2, 3, 4, 5])
            | 'Combine globally' >> beam.CombineGlobally(sum)
            | 'Print total' >> beam.Map(lambda x: print(f"Total: {x}"))
        )

print("Combine concept: Efficient aggregation operations")

## Exercise 3: Advanced Aggregation

In [ ]:
def advanced_aggregation():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        sales = pipeline | 'Create data' >> beam.Create(data)
        
        (
            sales
            | 'Extract product revenue' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Combine revenue' >> beam.CombinePerKey(sum)
            | 'Format revenue' >> beam.Map(
                lambda x: f"Product {x[0]}: ${x[1]:.2f}")
            | 'Print revenue' >> beam.Map(print)
        )
        
        (
            sales
            | 'Calculate transaction totals' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Calculate average' >> beam.CombineGlobally(
                lambda values: sum(values) / len(values) if values else 0)
            | 'Print average' >> beam.Map(
                lambda x: print(f"Average transaction: ${x:.2f}"))
        )
        
        (
            sales
            | 'Extract customer' >> beam.Map(lambda x: (x['customer_id'], 1))
            | 'Count transactions' >> beam.CombinePerKey(sum)
            | 'Format counts' >> beam.Map(
                lambda x: f"Customer {x[0]}: {x[1]} transactions")
            | 'Print counts' >> beam.Map(print)
        )

print("Performing advanced aggregations...")
advanced_aggregation()

## Summary

This solution demonstrates:
- Custom DoFn implementation for data enrichment
- ParDo transform usage for element-wise processing
- GroupByKey for grouping operations
- Combine transforms for efficient aggregations
- Real-world data processing patterns

These core transforms are fundamental building blocks for Apache Beam pipelines.